# Identifying Local Activity of White Terror Organizations

In this notebook, I demonstrate my process for identifying newspaper coverage of local activity of white terrorist organizations. This became my next step after identifying and tagging articles about local instances of lynchings (see 03_tag_local_reports.ipynb). In that previous process, I identified and tagged a total of 300 articles about local instances of lynchings. These articles ranged in length–some were full-page coverage of events, others were brief, one-paragraph updates about subsequent court proceedings, follow-up news, etc. This was an effective sample of the kinds of materials I was looking for. These 300 articles signal local racial violence and tension, but after reflecting on it, I realized that this sample would only help to train a model on a narrow type of racial violence reporting: that of public lynchings.

As I continued to reflect on the larger process, I decided to seek other forms of newspaper materials that signal threats of racial violence. I brainstormed for a while about what other kinds of local coverage I might be able to identify and that would diversify my signal of sundown towns and counties. I came up with a few ideas, the first of which I'm enacting here. It's identifying local activity of white terrorist organizations. The logic here is fairly straightforward: if there were known and reported activities of white terror organizations like the Ku Klux Klan, the Red Shirts, or the White League, then non-white people would be under threat. These senses of threat, I should reiterate, are the feature of sundown towns and counties I'm trying to identify in local newspaper coverage. So, in other words, in this notebook, I'm classifying local reports of white terror organizations as a signal of sundown towns and trying to identify them.

I could seek out this signal in newspapers in general, but I'm starting again with newspapers from places I've already classified as sundown towns and counties; that is, newspapers from counties where racially motivated lynchings occurred. Pulling local activity of white terror organizations from this localized data adds another verification layer to my overall process. Since I know these places became materially dangerous to non-white people, I know that any local coverage of racist or racially violent content was not merely posturing or empty threats. For non-white people in these places, the threats would have been phenomenological (experienced subjectively, embodied, felt), epistemological (experienced as local knowledge and passed along via newspapers), and material in that these threats culminated in real violence.

This step does not argue that KKK or other white terror organizations necessarily committed the local lynchings; although, to presume their involvement in some capacity would be entirely reasonable. But rather, this step in my larger process argues that any local activity of white terror organizations adds the pertinent phenomenological dimension of sundown towns because such activity is inherently threatening to non-white people.

All right, so that's how I got here, basically. But how to operationalize this identification of local white terror activity?

It took me a while, but eventually I conceived of the following steps:

1) identify mentions of white terror organizations: Ku Klux Klan, Red Shirts, and the White League in sundown county newspapers
2) narrow the results to only those that appear within close context to "local words" (a lexicon of words that signal local discourse)
3) hand-key these narrowed results as either local coverage of white terror organizations or not

This notebook doesn't actually complete all three steps–just the first two. I'll draft another notebook for the hand-keying process and results. But this notebook does ultimately narrow the results substantially. As you'll see, I end up with 263 potential references to local white terror organizations, a totally manageable number to review by hand.

So, let's go over things more granularly. As always, I started by importing necessary libraries and setting directories, paths, placeholders, etc.

In [ ]:
from pathlib import Path
from functools import partial
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
import pandas as pd
import os
import re
import bisect

BASE_DIR = Path('/Volumes/t5_evo_8tb/ChronAm/sundown_town_data')
SEARCH_TERMS = [
    'ku klux',
    'white cap',
    'red shirt',
    'white league',
]
OUTPUT_CSV = Path('white_terror_search_results.csv')

LEFT_WORDS = 50
RIGHT_WORDS = 100
MAX_WORKERS = 12
CHUNKSIZE = 100

SEARCH_PATTERNS = None

Next up, I put together some helper functions. I prompted Codex to put them together with a rigid set of logics in the prompt. The prompt was based off similar processes I've done in the past. Then I tweaked the Codex functions here and there and made sure the whole pipeline did what I needed.

The first function, token_variants, builds a series of potential OCR variants using regex. It takes the given search terms–"ku klux", "white cap", red shirt", or "white league"–and creates possible variations that are, at most, one character off. If any of the words in the search term are three characters or less, it also disregards them so it's not doing one-character variations for two or three letter words. This is an OCR correction approach I've worked with in the past. It's pretty conservative given that lots of Chronicling America's OCR is egregious, but it does catch a lot more potential search hits.

In [ ]:
def _token_variants(token: str) -> str:
    variants = [re.escape(token)]
    if len(token) >= 3:
        for i in range(len(token)):
            variants.append(re.escape(token[:i]) + '.' + re.escape(token[i + 1:]))
    variants = list(dict.fromkeys(variants))
    return '(?:' + '|'.join(variants) + ')'

This function works with token_variants by splitting search terms into tokens. Then, once the potential variants are compiled, it brings them back together into the search terms so they can be applied.


In [ ]:
def _compile_phrase_pattern(term: str) -> re.Pattern:
    parts = [p for p in str(term).split() if p]
    body = r'\W*'.join(_token_variants(part) for part in parts)
    return re.compile(rf'(?<!\w){body}(?!\w)', flags=re.IGNORECASE)

This function does some other light OCR cleaning. It accounts for words split across hyphenated line breaks and removes repeated spaces. Simple, common OCR issues.

In [ ]:
def _normalize_ocr_text(text: str) -> str:
    text = re.sub(r'(?<=\w)-\s+(?=\w)', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


The previous OCR search term function require tokenization and character counts per word. They also require word positions to identify sequences and, later, to calculate the clipping ranges. This function compiles all those things.


In [ ]:
def _word_spans(text: str):
    matches = list(re.finditer(r'\S+', text))
    words = [m.group(0) for m in matches]
    spans = [m.span() for m in matches]
    starts = [s for s, _ in spans]
    return words, spans, starts

This function creates a word index, making it possible to identify word positions and subsequently build clippings based on the words prior to, and following, positive search hits.


In [ ]:
def _char_to_word_idx(pos: int, spans, starts) -> int:
    i = bisect.bisect_right(starts, pos) - 1
    if i < 0:
        return 0
    n = len(spans)
    while i + 1 < n and spans[i][1] <= pos:
        i += 1
    return min(i, n - 1)

This function finds search hits and extracts the text surrounding them (the 50 words prior to the search hit and the 100 words afterwards). This extracted text becomes a clipping, a chunk of newspaper OCR text not unlike a traditional newspaper clipping. These clippings are used to help understand the context of positive search hits so it's easier to determine if they are relevant to our training set.

In [ ]:
def _extract_clippings(
    text: str,
    pattern: re.Pattern,
    left_words: int = 50,
    right_words: int = 100,
):
    matches = list(pattern.finditer(text))
    if not matches:
        return []

    words, spans, starts = _word_spans(text)
    if not words:
        return []

    out = []
    for m in matches:
        s_char, e_char = m.span()
        s_idx = _char_to_word_idx(s_char, spans, starts)
        e_idx = _char_to_word_idx(max(s_char, e_char - 1), spans, starts)

        left = max(0, s_idx - left_words)
        right = min(len(words), e_idx + 1 + right_words)

        out.append(
            {
                'matched_text': m.group(0),
                'clipping': ' '.join(words[left:right]),
            }
        )
    return out

I also needed to extract the Chron Am metadata from the directory paths I was searching from. If you remember, my whole process has been done on my local version of Chronicling America (see my [blog](https://matthewkollmer.com/how-im-backing-up-chronicling-america/) for more info). It has nested directories that contain info about the newspaper pages. To ensure I maintain that info in my data, I put together this function:

In [ ]:
def _path_fields(txt_path: Path, base_dir: Path) -> dict:
    rel_parts = txt_path.relative_to(base_dir).parts
    fields = {
        'lccn': None,
        'year': None,
        'month': None,
        'date': None,
        'ed': None,
        'seq': None,
    }

    if len(rel_parts) >= 8:
        fields.update(
            {
                'lccn': rel_parts[-7],
                'year': rel_parts[-6],
                'month': rel_parts[-5],
                'date': rel_parts[-4],
                'ed': rel_parts[-3],
                'seq': rel_parts[-2],
            }
        )

    return fields

This is a little efficiency step–it applies the regex patterns to the search terms (compile_phrase_pattern) and saves the results in a dictionary so the whole program doesn't need to redo that computationally intensive step over and over again.

In [ ]:
def _init_worker(search_terms):
    global SEARCH_PATTERNS
    SEARCH_PATTERNS = {
        term: _compile_phrase_pattern(term)
        for term in search_terms
    }

This function brings it all together. It scans a single OCR text file. It reads and normalizes the file, extracts metadata from the path, runs every precompiled search pattern against the text, and returns one row per clipping with the associated search term and Chron Am metadata.

In [ ]:
def _scan_one_file(
    txt_path_str: str,
    base_dir_str: str,
    left_words: int,
    right_words: int,
):
    global SEARCH_PATTERNS

    txt_path = Path(txt_path_str)
    base_dir = Path(base_dir_str)

    try:
        raw_text = txt_path.read_text(encoding='utf-8', errors='ignore')
    except OSError:
        return []

    text = _normalize_ocr_text(raw_text)
    if not text:
        return []

    meta = _path_fields(txt_path, base_dir)
    rows = []

    for search_term, pattern in SEARCH_PATTERNS.items():
        clips = _extract_clippings(
            text,
            pattern=pattern,
            left_words=left_words,
            right_words=right_words,
        )
        for clip in clips:
            rows.append(
                {
                    'search_term': search_term,
                    'lccn': meta['lccn'],
                    'year': meta['year'],
                    'month': meta['month'],
                    'date': meta['date'],
                    'ed': meta['ed'],
                    'seq': meta['seq'],
                    'ocr_path': str(txt_path),
                    'clipping': clip['clipping']
                }
            )

    return rows

To ensure I navigate and search every OCR text file, I also put togother this scanner that builds a file list so the search knows where every text file resides.

In [ ]:
def _iter_txt_files(base_dir: Path):
    for root, _, files in os.walk(base_dir):
        for fn in files:
            if fn.lower().endswith('.txt'):
                yield str(Path(root) / fn)


And finally, the main search pipeline. It gathers all OCR text files, configures the per-file worker, optionally runs the search in parallel, combines all returned rows into a DataFrame, and sorts the final results for stable output.

In [ ]:
def search_terms_with_clippings(
    base_dir,
    search_terms,
    left_words: int = 50,
    right_words: int = 100,
    max_workers: int = None,
    chunksize: int = 100,
) -> pd.DataFrame:
    base_dir = Path(base_dir)
    txt_paths = list(_iter_txt_files(base_dir))

    if not txt_paths:
        return pd.DataFrame(
            columns=[
                'search_term',
                'lccn',
                'year',
                'month',
                'date',
                'ed',
                'seq',
                'ocr_path',
                'clipping',
            ]
        )

    worker = partial(
        _scan_one_file,
        base_dir_str=str(base_dir),
        left_words=left_words,
        right_words=right_words,
    )

    all_rows = []

    if max_workers == 1:
        _init_worker(search_terms)
        for path in txt_paths:
            batch = worker(path)
            if batch:
                all_rows.extend(batch)
    else:
        executor_kwargs = {
            'max_workers': max_workers,
            'initializer': _init_worker,
            'initargs': (search_terms,),
        }
        if os.name != 'nt':
            executor_kwargs['mp_context'] = mp.get_context('fork')

        with ProcessPoolExecutor(**executor_kwargs) as ex:
            for batch in ex.map(worker, txt_paths, chunksize=chunksize):
                if batch:
                    all_rows.extend(batch)

    results = pd.DataFrame(
        all_rows,
        columns=[
            'search_term',
            'lccn',
            'year',
            'month',
            'date',
            'ed',
            'seq',
            'ocr_path',
            'clipping',
        ],
    )

    if not results.empty:
        results = results.sort_values(
            ['search_term', 'lccn', 'year', 'month', 'date', 'ed', 'seq', 'ocr_path'],
            kind='stable',
        ).reset_index(drop=True)

    return results

And now, I was ready to execute the whole program! No small thing. This code cell does just that. It ran for over an hour. It also resulted in 26,679 positive search results saved in white_terror_search_results.csv.

In [ ]:
if __name__ == '__main__':
    results_df = search_terms_with_clippings(
        base_dir=BASE_DIR,
        search_terms=SEARCH_TERMS,
        left_words=LEFT_WORDS,
        right_words=RIGHT_WORDS,
        max_workers=MAX_WORKERS,
        chunksize=CHUNKSIZE,
    )
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f'Wrote {len(results_df):,} rows to {OUTPUT_CSV}')

This was a lot of search results... Perhaps too much. I became even more suspicious after reading the data myself. I looked at dozens of hits for "red shirt" and "white cap" and realized lots were irrelevant. "Ku klux" was the predominate search hit, but there were lots of "red shirt" and "white cap" hits, too. To me, this meant I probably had a lot of false positives.

As I mulled that over, I also realized I had forgotten something, too. I needed the Chron Am urls to these newspaper pages. Oops! Almost forgot. To compile the urls, I put together this code:

In [ ]:
test_df = pd.read_csv('white_terror_search_results.csv')

seq_no = test_df['seq'].astype(str).str.extract(r'(\d+)', expand=False)
search_q = (
    test_df['search_term']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '+', regex=True)
)

month = test_df['month'].astype(str).str.zfill(2)
day = test_df['date'].astype(str).str.zfill(2)

test_df['chron_am_url'] = (
    'https://www.loc.gov/resource/'
    + test_df['lccn'].astype(str)
    + '/'
    + test_df['year'].astype(str)
    + '-'
    + month
    + '-'
    + day
    + '/'
    + test_df['ed'].astype(str)
    + '/?sp='
    + seq_no.fillna('')
    + '&q='
    + search_q
    + '&st=text&r=0,0,0,0'
)

test_df.to_csv('white_terror_search_results.csv', index=False)

Anyway, I now had a csv (white_terror_search_results.csv) with over 26,000 rows, more than I wanted to hand-tag and lots that were probably false positives (i.e., not referencing local instances of white terrorist organization activity). At this point, I needed to think of a way to narrow these results.

My first idea was to parse and tag these results using LLM labeling of presuppositions of local context. It would work like this: I would set up an LLM prompt that takes the clippings and creates a list of presuppositions identified by the LLM. These presuppositions would be simple statements generated by the LLM that attempt to describe the presuppositions of the clipping content. Then I'd parse these presuppositions in another iteration, using the LLM to label them as related to local context or not. Then I'd have to look through the results by hand, double-check, basically, and build a testing verification set, too.

I thought this would be an interesting approach, but the downside was that it would be computationally expensive. With two LLM prompts embedded in the process, I'd have to iterate over at least 50,000 API calls. That's way more than I wanted to spend. And while I was intrigued about this novel approach to narrowing my data (experimenting with presupposition labels), I wasn't sure how much it would narrow things down, or how accurately.

So I backed off the idea. Instead, I decided to parse and tag using lexicons of local words or phrases. It would work like this: I would create a lexicon of general local words (town, village, courthouse, county, etc.). Then I'd check word frequency of those words in the clippings and narrow candidates for hand-labeling a certain threshold of "local" words. This was rather rudimentary compared to the big LLM idea but it would be relatively straightforward and not expensive.

That settled it. I went with the simpler word frequency approach. My lexicon of "local" words was fairly intuitive, too. Since I was going to hand-label the results anyway, I didn't feel the need to make the lexicon any more scientific than what I would intuit as relevant "local" word signals. In other words, I just made up the lexicon myself (just words I personally associate with local context): 'town', 'courthouse', 'county', 'city', 'jail', 'mayor', 'sheriff', 'citizen', resident', 'neighbor', 'local', 'school', 'church', 'street', 'road'

With these words decided, I moved ahead, counting these words in the clippings:

In [ ]:
df = pd.read_csv('white_terror_search_results.csv')

local_words = [
    'town',
    'courthouse',
    'county',
    'city',
    'jail',
    'mayor',
    'sheriff',
    'citizen',
    'resident',
    'neighbor',
    'local',
    'school',
    'church',
    'street',
    'road',
]

pattern = re.compile(r'\b(?:%s)\b' % '|'.join(map(re.escape, local_words)), flags=re.IGNORECASE)

df['local_word_count'] = (
    df['clipping']
    .fillna('')
    .astype(str)
    .apply(lambda text: len(pattern.findall(text)))
)

df.to_csv('white_terror_search_results.csv', index=False)

Then I also remembered I had city/town, county, and state names for each newspaper! I could these words to the lexicon or design my threshold to incorporate them, too. I just had to match them to my Chron Am newspaper data (downloaded from the [Chron Am website](https://www.loc.gov/collections/chronicling-america/titles/)):

In [ ]:
newspapers = pd.read_csv('chron_am_newspapers.csv')

lookup = newspapers[['LCCN', 'County', 'City', 'State']].copy()
lookup['LCCN'] = lookup['LCCN'].astype(str).str.strip()
df['lccn'] = df['lccn'].astype(str).str.strip()

df = df.merge(
    lookup,
    left_on='lccn',
    right_on='LCCN',
    how='left'
)

df = df.rename(
    columns={
        'County': 'county',
        'City': 'city',
        'State': 'state',
    }
).drop(columns=['LCCN'])

df.to_csv('white_terror_search_results.csv', index=False)

Than I counted instances of the city, state, and county words in the clippings:

In [ ]:
def count_place_words(row):
    clipping = str(row.get('clipping', '') or '')

    # Collect words from county, city, and state for this row
    place_text = ' '.join(
        str(row.get(col, '') or '')
        for col in ['county', 'city', 'state']
    )

    # Extract distinct words, case-insensitive
    place_words = set(re.findall(r'\b\w+\b', place_text.lower()))
    if not place_words:
        return 0

    pattern = re.compile(
        r'\b(?:%s)\b' % '|'.join(map(re.escape, place_words)),
        flags=re.IGNORECASE
    )

    return len(pattern.findall(clipping))

df['local_place_references'] = df.apply(count_place_words, axis=1)

df.to_csv('white_terror_search_results.csv', index=False)

Okay, at this point, I was ready to calibrate a kind of threshold for what data I would review. The smaller the sample, the better, but if after hand-reviewing I didn't feel like I had identified enough articles about local activity of white terror organizations, I also wanted to recalibrate easily.

I came up with the following metric: I would hand-review any positive search hit that had four or more "local words" in its clipping and at least one mention of the relevant town/city, county, or state.

In [ ]:
terms = ['red shirt', 'ku klux', 'white league']

subset_df = df[
    (df['search_term'].isin(terms)) &
    (df['local_word_count'] >= 4) &
    (df['local_place_references'] >= 1)
].copy()

subset_df.to_csv('white_terror_to_review.csv', index=False)

subset_df

# toy with these parameters
# set as follows, yielded 171 rows for review:
    #(df['search_term'].eq('ku klux')) &
    #(df['local_word_count'] >= 4) &
    #(df['local_place_references'] >= 1)

That left me with 263 rows to review by hand. Not too shabby. I may end up coming back to this step, however, if the results are not fruitful. We'll see.